In [16]:
import requests
import pandas as pd
import os
from time import sleep

def extrair_ifdata_robusto(pasta_destino):
    os.makedirs(pasta_destino, exist_ok=True)
    
    tipo_inst = 1
    relatorios = {"1": "Ativo", "2": "Passivo", "3": "Resultado"}
    anos = range(2014, 2025)
    trimestres = ["03", "06", "09", "12"]
    
    print(f"Iniciando Extração de Alta Performance: {os.path.abspath(pasta_destino)}")
    print("-" * 60)

    # 1. A MÁGICA DA VELOCIDADE: Usar uma Sessão mantém a conexão TCP/SSL aberta
    session = requests.Session()
    session.headers.update({"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"})

    for ano in anos:
        for tri in trimestres:
            data_ref = int(f"{ano}{tri}")
            
            for cod_rel, nome_rel in relatorios.items():
                nome_arquivo = f"IFDATA_{data_ref}_{nome_rel}.csv"
                caminho_completo = os.path.join(pasta_destino, nome_arquivo)
                
                # Pula o que você já conseguiu baixar!
                if os.path.exists(caminho_completo):
                    print(f"[PULANDO] {nome_arquivo} já existe.")
                    continue
                
                url = (
                    f"https://olinda.bcb.gov.br/olinda/servico/IFDATA/versao/v1/odata/"
                    f"IfDataValores(AnoMes=@AnoMes,TipoInstituicao=@TipoInstituicao,Relatorio=@Relatorio)"
                    f"?@AnoMes={data_ref}&@TipoInstituicao={tipo_inst}&@Relatorio='{cod_rel}'&$format=json"
                )
                
                sucesso = False
                
                # 2. SISTEMA DE RETENTATIVAS: Tenta até 3 vezes antes de acusar falha
                for tentativa in range(1, 4):
                    try:
                        # 3. AUMENTO DO TIMEOUT: Damos 120 segundos para o servidor do BCB pensar
                        resposta = session.get(url, timeout=120)
                        
                        if resposta.status_code == 200:
                            dados = resposta.json().get('value', [])
                            if dados:
                                df = pd.DataFrame(dados)
                                df.to_csv(caminho_completo, index=False, sep=';', encoding='utf-8')
                                print(f"[OK] Sucesso: {data_ref} - {nome_rel} ({len(df)} linhas)")
                            else:
                                print(f"[-] Vazio: {data_ref} - {nome_rel}")
                                
                            sucesso = True
                            break # Deu certo! Sai do loop de tentativas e vai pro próximo arquivo
                            
                        else:
                            print(f"[Erro {resposta.status_code}] Tentativa {tentativa}/3 para {nome_rel}_{data_ref}")
                            sleep(3) # Pausa antes de tentar de novo
                            
                    except requests.exceptions.ReadTimeout:
                        print(f"[Timeout] BCB lento na tentativa {tentativa}/3 ({nome_rel}_{data_ref}). Retentando...")
                        sleep(5)
                    except requests.exceptions.RequestException as e:
                        print(f"[Falha de Rede] Tentativa {tentativa}/3: {e}")
                        sleep(5)
                
                if not sucesso:
                    print(f"[DESISTÊNCIA] Não foi possível baixar {nome_arquivo}. Tente em outro horário.")
                
                sleep(0.5)

    print("\nProcesso concluído!")

# ==========================================
# Executar
# ==========================================
extrair_ifdata_robusto('./dados_bcb_olinda')

Iniciando Extração de Alta Performance: /home/inovar-para-pessoas-negras/Área de trabalho/gustavo/bank-distress-foretoken/dados_bcb_olinda
------------------------------------------------------------
[OK] Sucesso: 201403 - Ativo (9630 linhas)
[PULANDO] IFDATA_201403_Passivo.csv já existe.
[PULANDO] IFDATA_201403_Resultado.csv já existe.
[OK] Sucesso: 201406 - Ativo (9558 linhas)
[OK] Sucesso: 201406 - Passivo (27081 linhas)
[OK] Sucesso: 201406 - Resultado (33453 linhas)
[OK] Sucesso: 201409 - Ativo (9540 linhas)
[OK] Sucesso: 201409 - Passivo (27030 linhas)
[OK] Sucesso: 201409 - Resultado (33390 linhas)
[OK] Sucesso: 201412 - Ativo (9432 linhas)
[PULANDO] IFDATA_201412_Passivo.csv já existe.
[OK] Sucesso: 201412 - Resultado (33012 linhas)
[OK] Sucesso: 201503 - Ativo (14013 linhas)
[OK] Sucesso: 201503 - Passivo (26469 linhas)
[OK] Sucesso: 201503 - Resultado (32697 linhas)
[OK] Sucesso: 201506 - Ativo (13842 linhas)
[OK] Sucesso: 201506 - Passivo (26146 linhas)
[OK] Sucesso: 201506 

In [ ]:
import pandas as pd
import glob
import os

def processar_e_unir_ifdata(pasta_dados):
    print("Iniciando a leitura de todos os arquivos...")
    
    arquivos = glob.glob(os.path.join(pasta_dados, "IFDATA_*.csv"))
    
    if not arquivos:
        print("Nenhum arquivo encontrado na pasta.")
        return None

    lista_dfs = []

    # 1. Lê todos os arquivos e coloca numa lista
    for arquivo in arquivos:
        print(f"Lendo: {os.path.basename(arquivo)}")
        df = pd.read_csv(arquivo, sep=';', dtype={'CodInst': str, 'AnoMes': str})
        lista_dfs.append(df)

    print("\nEmpilhando os dados (Concatenando)...")
    # 2. Junta todos os arquivos um embaixo do outro
    df_completo = pd.concat(lista_dfs, ignore_index=True)

    print("Limpando e estruturando os nomes das contas contábeis...")
    # 3. Limpeza e prefixos
    if 'NomeColuna' in df_completo.columns:
        df_completo['NomeColuna'] = df_completo['NomeColuna'].str.replace('\n', ' ', regex=False)
        df_completo['NomeColuna'] = df_completo['NomeColuna'].str.replace(r'\s+', ' ', regex=True).str.strip()
        
        if 'NomeRelatorio' in df_completo.columns:
            df_completo['NomeColuna'] = df_completo['NomeRelatorio'] + " - " + df_completo['NomeColuna']

    print("Girando a tabela para o formato de Inteligência Artificial (Pivot)...")
    # 4. O PIVOT ÚNICO E PODEROSO: Transforma as contas em colunas de uma só vez
    # Como todos os dados já estão empilhados, não haverá colisão.
    df_mestre = df_completo.pivot_table(
        index=['CodInst', 'AnoMes'], 
        columns='NomeColuna', 
        values='Saldo', 
        aggfunc='first'
    ).reset_index()

    # Preenche o que o banco não reportou com 0
    df_mestre = df_mestre.fillna(0)

    # 5. Salva o Dataset
    caminho_saida = os.path.join(pasta_dados, "DATASET_MESTRE_IA.csv")
    df_mestre.to_csv(caminho_saida, index=False, sep=';', encoding='utf-8-sig')
    
    print("-" * 60)
    print(f"SUCESSO! Tabela final criada com {df_mestre.shape[0]} registros e {df_mestre.shape[1]} colunas contábeis.")
    print(f"Arquivo salvo em: {caminho_saida}")
    
    return df_mestre

# ==========================================
# Executar
# ==========================================
dataset_final = processar_e_unir_ifdata('./dados_bcb_olinda')

Iniciando o processamento e tratamento de colisões...
Lendo: IFDATA_202103_Passivo.csv
Lendo: IFDATA_202212_Resultado.csv
Lendo: IFDATA_201506_Ativo.csv
Lendo: IFDATA_202403_Resultado.csv
Lendo: IFDATA_202312_Resultado.csv
Lendo: IFDATA_202406_Passivo.csv
Lendo: IFDATA_201909_Ativo.csv
Lendo: IFDATA_201603_Passivo.csv
Lendo: IFDATA_201809_Resultado.csv
Lendo: IFDATA_202006_Resultado.csv
Lendo: IFDATA_202206_Resultado.csv
Lendo: IFDATA_202409_Ativo.csv
Lendo: IFDATA_201709_Ativo.csv
Lendo: IFDATA_201503_Ativo.csv
Lendo: IFDATA_201612_Ativo.csv
Lendo: IFDATA_201709_Passivo.csv
Lendo: IFDATA_202412_Ativo.csv
Lendo: IFDATA_201512_Resultado.csv
Lendo: IFDATA_202106_Ativo.csv
Lendo: IFDATA_202103_Ativo.csv
Lendo: IFDATA_202409_Passivo.csv
Lendo: IFDATA_201412_Passivo.csv
Lendo: IFDATA_201806_Ativo.csv
Lendo: IFDATA_201506_Resultado.csv
Lendo: IFDATA_202003_Passivo.csv
Lendo: IFDATA_201806_Passivo.csv
Lendo: IFDATA_202209_Resultado.csv
Lendo: IFDATA_201403_Resultado.csv
Lendo: IFDATA_201409_P

MergeError: Passing 'suffixes' which cause duplicate columns {'Passivo - Depósitos a Prazo (a4)_y', 'Passivo - Depósitos de Poupança (a2)_y', 'Passivo - Letras de Crédito Imobiliário (c1)_y', 'Passivo - Depósitos de Poupança (a2)_x', 'Passivo - Recursos de Aceites e Emissão de Títulos (c)_y', 'Passivo - Depósitos Interfinanceiros (a3)_y', 'Passivo - Letras de Crédito do Agronegócio (c2)_x', 'Passivo - Outras Obrigações (g)_x', 'Passivo - Passivo Circulante e Exigível a Longo Prazo (h) = (e) + (f) + (g)_y', 'Passivo - Letras Financeiras (c3)_y', 'Passivo - Depósitos à Vista (a1)_x', 'Passivo - Instrumentos Derivativos (f)_x', 'Passivo - Obrigações por Operações Compromissadas (b)_y', 'Passivo - Depósitos à Vista (a1)_y', 'Passivo - Recursos de Aceites e Emissão de Títulos (c)_x', 'Passivo - Depósitos a Prazo (a4)_x', 'Passivo - Letras Financeiras (c3)_x', 'Passivo - Outros Recursos de Aceites e Emissão de Títulos (c5)_x', 'Passivo - Obrigações por Empréstimos e Repasses (d)_x', 'Passivo

In [ ]:
import torch
import pandas as pd
import glob
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data

# ---------------------------------------------------------
# 1. Definição da Arquitetura do Modelo (GNN)
# ---------------------------------------------------------
class BankBankruptcyGCN(torch.nn.Module):
    def __init__(self, num_node_features, hidden_channels):
        super(BankBankruptcyGCN, self).__init__()
        # Primeira camada convolucional em grafo
        self.conv1 = GCNConv(num_node_features, hidden_channels)
        # Segunda camada convolucional
        self.conv2 = GCNConv(hidden_channels, hidden_channels // 2)
        # Camada linear de saída para classificação binária (0 = Saudável, 1 = Falência)
        self.out = torch.nn.Linear(hidden_channels // 2, 2)

    def forward(self, x, edge_index):
        # x: Matriz de atributos dos nós (Bancos x Variáveis do COSIF)
        # edge_index: Estrutura do grafo (Similaridade entre bancos)
        
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.3, training=self.training) # Previne overfitting
        
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        
        x = self.out(x)
        return x # Retorna os logits brutos



motherfolder = r"/home/inovar-para-pessoas-negras/Área de trabalho/gustavo/bank-distress-foretoken/dres"
minorfolders = [f for f in glob.glob(motherfolder = "/*.csv")]

summarized_csv = pd.Dataframe(columns = [])

# leitura dos arquivos e obtenção das informações
summary_table = pd.DataFrame(columns = ['Instituição';'Código';'TCB';'SR';'TD';'TC';'Cidade';'UF';'Data';'Resultado de Intermediação Financeira'])

for file in minorfolders:
    try:
        f = pd.read_csv(file, sep = ";") 
        info = [file.split(';;;;;;;;;;;;;;')[len(file.split(';;;;;;;;;;;;;;'))-1], 
                f.shape[0], 
                f.shape[1]-1, 
                f['Data'].min(),
                f['Data'].max(), 
                f.isna().sum().sum()] 
        
        new_info = pd.DataFrame([info], columns = ['Instituição';'Código';'TCB';'SR';'TD';'TC';'Cidade';'UF';'Data';'Resultado de Intermediação Financeira'])
        summary_table = pd.concat([summary_table, new_info], ignore_index = True)
    except:
        pass

#num_bancos = 100
#num_features = 15

# Tensor de features dos nós (X)
#x = torch.rand((num_bancos, num_features), dtype=torch.float)

# Tensor de arestas (Ex: matriz 2xN definindo conexões de similaridade)
# Aqui, o banco 0 está conectado ao 1, o 1 ao 2, etc.
edge_index = torch.tensor([[0, 1, 1, 2, 50, 60],
                           [1, 0, 2, 1, 60, 50]], dtype=torch.long)

# Rótulos das classes (Y): 0 para saudável, 1 para falência
# Desbalanceamento severo: apenas 5 bancos faliram neste cenário
y = torch.zeros(num_bancos, dtype=torch.long)
y[:5] = 1 

# Empacotando no formato do PyTorch Geometric
data = Data(x=x, edge_index=edge_index, y=y)

# ---------------------------------------------------------
# 3. Configuração e Loop de Treinamento
# ---------------------------------------------------------
model = BankBankruptcyGCN(num_node_features=num_features, hidden_channels=32)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

# Como falência é raro, aplicamos pesos na função de perda (Weighted Cross Entropy)
# Dá muito mais peso à classe 1 (Falência) do que à classe 0 (Saudável)
pesos_classes = torch.tensor([1.0, 15.0]) 
criterion = torch.nn.CrossEntropyLoss(weight=pesos_classes)

def train():
    model.train()
    optimizer.zero_grad()  
    out = model(data.x, data.edge_index)  
    loss = criterion(out, data.y)  
    loss.backward()  
    optimizer.step()  
    return loss.item()

# Executando algumas épocas
for epoch in range(1, 101):
    loss = train()
    if epoch % 20 == 0:
        print(f'Época: {epoch:03d}, Loss: {loss:.4f}')